In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import re
import sys

from src.neuralnet import NeuralNetwork
class NeuralNetworkTracked(NeuralNetwork):
    pass
sys.modules["__main__"].NeuralNetworkTracked = NeuralNetworkTracked

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATH_DATASET_TESTE = "data/subm2.csv"
df_teste = pd.read_csv(PATH_DATASET_TESTE, sep=None, engine="python")

# Normalizar colunas para evitar problemas de BOM/capitalização
df_teste.columns = [str(c).replace("\ufeff", "").strip() for c in df_teste.columns]
cols_lower = {c.lower(): c for c in df_teste.columns}
if "id" in cols_lower and "ID" not in df_teste.columns:
    df_teste = df_teste.rename(columns={cols_lower["id"]: "ID"})
if "text" in cols_lower and "Text" not in df_teste.columns:
    df_teste = df_teste.rename(columns={cols_lower["text"]: "Text"})
if "Text" not in df_teste.columns:
    raise ValueError("Coluna 'Text' não encontrada no ficheiro de teste.")
if "ID" not in df_teste.columns:
    df_teste.insert(0, "ID", np.arange(1, len(df_teste) + 1, dtype=np.int64))
id_like = [c for c in df_teste.columns if c.lower() == "id"]
if len(id_like) > 1:
    keep = id_like[0]
    df_teste = df_teste.drop(columns=id_like[1:])
    if keep != "ID":
        df_teste = df_teste.rename(columns={keep: "ID"})

with open("modelo_numpy_artefactos.pkl", "rb") as f:
    le = pickle.load(f)["label_encoder"]

with open("pytorch_vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=2,
                          batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

model_pt = GRUClassifier(len(vocab), 128, 128, len(le.classes_)).to(device)
model_pt.load_state_dict(torch.load("modelo_pytorch_gru.pth", map_location=device))
model_pt.eval()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()

def encode_pt(text, vocab, max_len=120):
    tokens = clean_text(text)
    ids = [vocab.get(token, vocab["<unk>"]) for token in tokens][:max_len]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids

preds_pt_idx = []
with torch.no_grad():
    for text in df_teste["Text"].values:
        x = torch.tensor([encode_pt(text, vocab)], dtype=torch.long).to(device)
        output = model_pt(x)
        preds_pt_idx.append(torch.argmax(output, dim=1).item())

df_teste["Labels"] = le.inverse_transform(preds_pt_idx)

NOME_FICHEIRO_SAIDA = "subm2-g14-MEI-B.csv"
df_saida = df_teste[["ID", "Text", "Labels"]]
df_saida.to_csv(NOME_FICHEIRO_SAIDA, index=False, sep=";")

print(f"Previsões concluídas e guardadas em {NOME_FICHEIRO_SAIDA}")
df_saida.head()

Previsões concluídas e guardadas em subm2-g14-MEI-B.csv


,ID,Text,Labels
0,D2-101,Microbial mats of coexisting bacteria and arch...,Human
1,D2-102,The origin of life on Earth remains a complex ...,Human
2,D2-103,Estimates of the time at which life arose on E...,OpenAI
3,D2-104,Life on Earth emerged roughly 3.8-4 billion ye...,OpenAI
4,D2-105,Black holes predominantly form from the catast...,OpenAI
